In [1]:
from xgboost import XGBClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost as xgb

In [2]:
train = pd.read_csv('compe1/train.csv')
test = pd.read_csv('compe1/test.csv')
train_ID = train['id']
test_ID = test['id']
train.drop('id', axis=1, inplace=True)
test.drop('id', axis=1, inplace=True)


In [3]:
print(train.shape)
print(test.shape)

(18524, 8)
(6175, 7)


In [4]:
missing = train.isnull().sum()
missing = missing[missing > 0]
print(missing)

Time_spent_Alone             1190
Stage_fear                   1893
Social_event_attendance      1180
Going_outside                1466
Drained_after_socializing    1149
Friends_circle_size          1054
Post_frequency               1264
dtype: int64


In [5]:
missing = test.isnull().sum()
missing = missing[missing > 0]
print(missing)

Time_spent_Alone             425
Stage_fear                   598
Social_event_attendance      397
Going_outside                466
Drained_after_socializing    432
Friends_circle_size          350
Post_frequency               408
dtype: int64


In [6]:

y = train.Personality.reset_index(drop=True)
train = train.drop(['Personality'], axis=1)
train.columns
print(train.shape)
print(test.shape)

(18524, 7)
(6175, 7)


In [7]:
bool_list = ['Stage_fear', 'Drained_after_socializing']
for i in bool_list:
    #train[i].fillna(-1, inplace=True)
    train[i] = train[i].replace({'Yes': 1, 'No': 0})
    test[i] = test[i].replace({'Yes': 1, 'No': 0})

not_bool = [col for col in train.columns if col not in bool_list]
#for i in not_bool:
    #train[i] = train[i].fillna(train[i].median())

y = y.map({'Extrovert': 1, 'Introvert': 0})

def construct_features(df):
    import numpy as np

    # 布尔映射为正负1
    fear_map = {'Yes': -1, 'No': 1}
    drain_map = {'Yes': -1, 'No': 1}
    bin_labels = [0, 1, 2, 3]  # 四分位离散标签

    df['Stage_fear_sign'] = df['Stage_fear'].map(fear_map)
    df['Drained_social_sign'] = df['Drained_after_socializing'].map(drain_map)

    # 缺失计数（仍然有用）
    df['Missing_count'] = df.isnull().sum(axis=1)

    # 允许缺失参与表达
    df['Social_energy'] = (
        df['Social_event_attendance'] + 
        df['Going_outside'] + 
        df['Drained_social_sign']
    )

    df['Fear_x_Social'] = (
        df['Stage_fear_sign'] * df['Social_event_attendance']
    )

    df['Online_social_ratio'] = (
        df['Post_frequency'] / (df['Friends_circle_size'] + 1)
    )

    df['Alone_normalized'] = (
        df['Time_spent_Alone'] / (df['Going_outside'] + 1)
    )

    df['Is_low_post'] = (df['Post_frequency'] < 3).astype('Int64')  # Int64 支持 NaN

    df['Is_unbalanced_social'] = (
        (df['Friends_circle_size'] < 3) & (df['Social_event_attendance'] > 6)
    ).astype('Int64')

    # 分箱，qcut 会自动跳过 NaN
    try:
        df['Bin_Social_event_attendance'] = pd.qcut(
            df['Social_event_attendance'],
            q=4,
            labels=bin_labels,
            duplicates='drop'
        ).astype('Int64')
    except Exception:
        df['Bin_Social_event_attendance'] = np.nan

    return df

train = construct_features(train)
test = construct_features(test)
print(train.shape)
print(test.shape)
drop_features = [
    'Social_energy',
    'Fear_x_Social',
    'Stage_fear_sign',
    'Drained_social_sign',
    'Is_unbalanced_social'
]
train.drop(columns=drop_features, inplace=True)
test.drop(columns=drop_features, inplace=True)

print(train.shape)
print(test.shape)

(18524, 17)
(6175, 17)
(18524, 12)
(6175, 12)


/tmp/ipykernel_43290/4189038658.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train[i] = train[i].replace({'Yes': 1, 'No': 0})
/tmp/ipykernel_43290/4189038658.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test[i] = test[i].replace({'Yes': 1, 'No': 0})


In [8]:

X_train, X_val, y_train, y_val = train_test_split(train, y, test_size=0.2, random_state=42)

model = XGBClassifier(n_estimators=4, 
                      max_depth=4, learning_rate=0.5, 
                      objective='binary:logistic',
)

model.fit(X_train, y_train)
pred = model.predict(X_val)

In [9]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_val, pred)
print(accuracy)

0.9686909581646423


In [10]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print("最佳参数：", grid.best_params_)
print("最佳准确率：", grid.best_score_)


最佳参数： {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 200}
最佳准确率： 0.9696339423287572


In [11]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_val)

from sklearn.metrics import accuracy_score
print("验证集准确率：", accuracy_score(y_val, y_pred))


验证集准确率： 0.968421052631579


In [12]:
# 用最佳参数构建新模型
final_model = RandomForestClassifier(**grid.best_params_)
final_model.fit(train, y)

# 在验证集上预测
y_pred = final_model.predict(test)
y_pred = pd.Series(y_pred).map({1: 'Extrovert', 0: 'Introvert'})

submission = pd.DataFrame({
        "id": test_ID,
        'Personality': y_pred,
    })
submission.to_csv("submission11.csv", index=False)

In [13]:
from sklearn.model_selection import cross_val_score
import optuna

def objective_xgb(trial):
    params = {
        "verbosity": 0,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
    }

    model = xgb.XGBClassifier(**params, use_label_encoder=False)
    scores = cross_val_score(model, train, y, cv=5, scoring='accuracy')
    return scores.mean()

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=50)

print("Best XGBoost parameters:", study_xgb.best_params)

[I 2025-07-19 11:56:49,225] A new study created in memory with name: no-name-06a9b747-ce55-44ec-a275-f7594abfc6b3
[I 2025-07-19 11:56:49,697] Trial 0 finished with value: 0.9691753744720664 and parameters: {'max_depth': 7, 'learning_rate': 0.16431724494348213, 'n_estimators': 169, 'gamma': 1.3972501434138496, 'min_child_weight': 9, 'subsample': 0.9711283487966863, 'colsample_bytree': 0.7404443222445882, 'lambda': 0.00440034507768299, 'alpha': 0.006030532611946778}. Best is trial 0 with value: 0.9691753744720664.
[I 2025-07-19 11:56:52,422] Trial 1 finished with value: 0.9692833804064905 and parameters: {'max_depth': 6, 'learning_rate': 0.012752335347038568, 'n_estimators': 655, 'gamma': 0.2338797267428333, 'min_child_weight': 3, 'subsample': 0.6110911072848457, 'colsample_bytree': 0.5390187672129996, 'lambda': 0.01820949520010475, 'alpha': 0.04061892550825681}. Best is trial 1 with value: 0.9692833804064905.
[I 2025-07-19 11:56:53,660] Trial 2 finished with value: 0.969229384726145 and

Best XGBoost parameters: {'max_depth': 4, 'learning_rate': 0.04058675002042848, 'n_estimators': 428, 'gamma': 3.7185588554134625, 'min_child_weight': 2, 'subsample': 0.9451194963062347, 'colsample_bytree': 0.627500972051914, 'lambda': 0.008378838774673122, 'alpha': 1.0763099581037079}


In [14]:
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

def objective_cat(trial):
    params = {
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'iterations': trial.suggest_int('iterations', 100, 500),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'bootstrap_type': trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS']),
        'random_state': 42,
        'verbose': 0
    }

    model = CatBoostClassifier(**params)
    score = cross_val_score(model, train, y, cv=5, scoring='accuracy').mean()
    return score

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=50)

print("最佳参数（CatBoost）：", study_cat.best_params)


[I 2025-07-19 11:57:44,487] A new study created in memory with name: no-name-2f1bbac7-fb68-4082-afec-cafdc7357574
[I 2025-07-19 11:57:45,326] Trial 0 finished with value: 0.9689594646193486 and parameters: {'depth': 4, 'learning_rate': 0.027618512688743053, 'iterations': 199, 'l2_leaf_reg': 3.7650742496935585, 'bootstrap_type': 'Bernoulli'}. Best is trial 0 with value: 0.9689594646193486.
[I 2025-07-19 11:57:48,073] Trial 1 finished with value: 0.9663682257646109 and parameters: {'depth': 10, 'learning_rate': 0.18595150812568895, 'iterations': 135, 'l2_leaf_reg': 0.042815542881133345, 'bootstrap_type': 'Bernoulli'}. Best is trial 0 with value: 0.9689594646193486.
[I 2025-07-19 11:57:48,889] Trial 2 finished with value: 0.9690134748734271 and parameters: {'depth': 7, 'learning_rate': 0.2814165904477246, 'iterations': 106, 'l2_leaf_reg': 9.794847694838579, 'bootstrap_type': 'MVS'}. Best is trial 2 with value: 0.9690134748734271.
[I 2025-07-19 11:57:49,460] Trial 3 finished with value: 0.

最佳参数（CatBoost）： {'depth': 7, 'learning_rate': 0.2814165904477246, 'iterations': 106, 'l2_leaf_reg': 9.794847694838579, 'bootstrap_type': 'MVS'}


In [15]:
def objective_lgb(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42
    }

    model = LGBMClassifier(**params)
    score = cross_val_score(model, train, y, cv=5, scoring='accuracy').mean()
    return score

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=50)

print("最佳参数（LightGBM）：", study_lgb.best_params)

[I 2025-07-19 12:00:29,431] A new study created in memory with name: no-name-e62b7c8d-f844-469c-9d0f-8c8de326d6a8


[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000508 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 222
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.739524 -> initscore=1.043494
[LightGBM] [Info] Start training from score 1.043494
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

[I 2025-07-19 12:00:30,033] Trial 0 finished with value: 0.9693913280459829 and parameters: {'num_leaves': 73, 'max_depth': 3, 'learning_rate': 0.09403863066704922, 'n_estimators': 299, 'min_child_samples': 33, 'subsample': 0.7883630311632466, 'colsample_bytree': 0.9054362165166503}. Best is trial 0 with value: 0.9693913280459829.


[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000395 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 222
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.739524 -> initscore=1.043494
[LightGBM] [Info] Start training from score 1.043494
[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000357 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 222
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 12
[LightGBM] [Info] [binar

[I 2025-07-19 12:00:31,675] Trial 1 finished with value: 0.9614017162027848 and parameters: {'num_leaves': 40, 'max_depth': 11, 'learning_rate': 0.21369261613256207, 'n_estimators': 416, 'min_child_samples': 16, 'subsample': 0.5297381050167953, 'colsample_bytree': 0.9046448548559924}. Best is trial 0 with value: 0.9693913280459829.


...

[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000379 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 222
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.739524 -> initscore=1.043494
[LightGBM] [Info] Start training from score 1.043494
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

[I 2025-07-19 12:01:16,975] Trial 48 finished with value: 0.9693913426197159 and parameters: {'num_leaves': 71, 'max_depth': 3, 'learning_rate': 0.09335209195494773, 'n_estimators': 181, 'min_child_samples': 30, 'subsample': 0.5434701247407725, 'colsample_bytree': 0.9446969472112134}. Best is trial 41 with value: 0.9693913717671816.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000298 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 223
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.739524 -> initscore=1.043494
[LightGBM] [Info] Start training from score 1.043494
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

[I 2025-07-19 12:01:18,078] Trial 49 finished with value: 0.9677178408723254 and parameters: {'num_leaves': 70, 'max_depth': 9, 'learning_rate': 0.09694319251666124, 'n_estimators': 185, 'min_child_samples': 30, 'subsample': 0.5246706431253126, 'colsample_bytree': 0.859354405424201}. Best is trial 41 with value: 0.9693913717671816.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [16]:
xgb_model = xgb.XGBClassifier(**study_xgb.best_params, use_label_encoder=False)
cat_model = CatBoostClassifier(**study_cat.best_params)
lgb_model = LGBMClassifier(**study_lgb.best_params)
rf_model = RandomForestClassifier(**grid.best_params_)


xgb_model.fit(X_train, y_train)       
cat_model.fit(X_train, y_train)       
lgb_model.fit(X_train, y_train)      
rf_model.fit(X_train, y_train)

print("Finished")

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [12:01:18] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0:	learn: 0.4893616	total: 1.79ms	remaining: 188ms
1:	learn: 0.3678428	total: 3.44ms	remaining: 179ms
2:	learn: 0.2915337	total: 4.94ms	remaining: 170ms
3:	learn: 0.2404997	total: 7.43ms	remaining: 190ms
4:	learn: 0.2058363	total: 8.27ms	remaining: 167ms
5:	learn: 0.1825248	total: 9.53ms	remaining: 159ms
6:	learn: 0.1664121	total: 10.9ms	remaining: 154ms
7:	learn: 0.1551359	total: 12.2ms	remaining: 149ms
8:	learn: 0.1465304	total: 13.1ms	remaining: 141ms
9:	learn: 0.1406897	total: 14.4ms	remaining: 138ms
10:	learn: 0.1365564	total: 15.6ms	remaining: 135ms
11:	learn: 0.1337150	total: 19ms	remaining: 149ms
12:	learn: 0.1318731	total: 20.4ms	remaining: 146ms
13:	learn: 0.1305866	total: 22.3ms	remaining: 147ms
14:	learn: 0.1296101	total: 23.8ms	remaining: 145ms
15:	learn: 0.1288405	total: 25.4ms	remaining: 143ms
16:	learn: 0.1282616	total: 26.8ms	remaining: 140ms
17:	learn: 0.1275869	total: 28.4ms	remaining: 139ms
18:	learn: 0.1273324	total: 29.3ms	remaining: 134ms
19:	learn: 0.1268872	tot

In [17]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(
    estimators=[
        ('cat', cat_model),
        ('lgb', lgb_model),
        ('xgb', xgb_model),
        ('rf', rf_model)
    ],
    voting='soft'  # or 'hard'
)

voting_clf.fit(X_train, y_train)

y_pred = voting_clf.predict(X_val)
print("投票器准确率：", accuracy_score(y_val, y_pred))

0:	learn: 0.4893616	total: 1.57ms	remaining: 165ms
1:	learn: 0.3678428	total: 2.97ms	remaining: 154ms
2:	learn: 0.2915337	total: 4.36ms	remaining: 150ms
3:	learn: 0.2404997	total: 5.46ms	remaining: 139ms
4:	learn: 0.2058363	total: 6.29ms	remaining: 127ms
5:	learn: 0.1825248	total: 7.65ms	remaining: 127ms
6:	learn: 0.1664121	total: 9.09ms	remaining: 129ms
7:	learn: 0.1551359	total: 10.5ms	remaining: 129ms
8:	learn: 0.1465304	total: 11.5ms	remaining: 124ms
9:	learn: 0.1406897	total: 13.2ms	remaining: 127ms
10:	learn: 0.1365564	total: 14.7ms	remaining: 127ms
11:	learn: 0.1337150	total: 16.2ms	remaining: 127ms
12:	learn: 0.1318731	total: 17.9ms	remaining: 128ms
13:	learn: 0.1305866	total: 19.3ms	remaining: 127ms
14:	learn: 0.1296101	total: 20.7ms	remaining: 126ms
15:	learn: 0.1288405	total: 22.2ms	remaining: 125ms
16:	learn: 0.1282616	total: 23.6ms	remaining: 123ms
17:	learn: 0.1275869	total: 24.9ms	remaining: 122ms
18:	learn: 0.1273324	total: 25.7ms	remaining: 118ms
19:	learn: 0.1268872	t

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [12:01:19] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


投票器准确率： 0.9686909581646423


In [18]:
xgb_model = xgb.XGBClassifier(**study_xgb.best_params, use_label_encoder=False)
cat_model = CatBoostClassifier(**study_cat.best_params)
lgb_model = LGBMClassifier(**study_lgb.best_params)
rf_model = RandomForestClassifier(**grid.best_params_)


xgb_model.fit(train, y)       
cat_model.fit(train, y)       
lgb_model.fit(train, y)      
rf_model.fit(train, y)

print("Finished")

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [12:01:21] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0:	learn: 0.4870800	total: 1.56ms	remaining: 164ms
1:	learn: 0.3659073	total: 2.63ms	remaining: 137ms
2:	learn: 0.2898821	total: 3.95ms	remaining: 136ms
3:	learn: 0.2394434	total: 5.05ms	remaining: 129ms
4:	learn: 0.2051688	total: 5.99ms	remaining: 121ms
5:	learn: 0.1812125	total: 8.45ms	remaining: 141ms
6:	learn: 0.1648519	total: 9.78ms	remaining: 138ms
7:	learn: 0.1532523	total: 11ms	remaining: 135ms
8:	learn: 0.1455545	total: 12.4ms	remaining: 133ms
9:	learn: 0.1404007	total: 14.9ms	remaining: 143ms
10:	learn: 0.1367551	total: 17ms	remaining: 146ms
11:	learn: 0.1343020	total: 18.4ms	remaining: 144ms
12:	learn: 0.1324961	total: 19.8ms	remaining: 142ms
13:	learn: 0.1312941	total: 21.2ms	remaining: 139ms
14:	learn: 0.1305300	total: 22.7ms	remaining: 137ms
15:	learn: 0.1300211	total: 24ms	remaining: 135ms
16:	learn: 0.1295588	total: 25.5ms	remaining: 133ms
17:	learn: 0.1287819	total: 26.9ms	remaining: 132ms
18:	learn: 0.1285440	total: 27.9ms	remaining: 128ms
19:	learn: 0.1285420	total: 

In [22]:
def best_features(model, num=50):
    # 获取特征重要性
    importance = model.feature_importances_
    features = X_train.columns
    
    # 打包成 DataFrame
    feat_imp = pd.DataFrame({
        'feature': features,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    # 显示前 30 个最重要的特征
    print(feat_imp.head(num))

best_features(xgb_model, 17)
best_features(cat_model, 17)
best_features(lgb_model, 17)
best_features(rf_model, 17)

                        feature  importance
4     Drained_after_socializing    0.399877
1                    Stage_fear    0.248912
0              Time_spent_Alone    0.128681
11  Bin_Social_event_attendance    0.092182
10                  Is_low_post    0.041427
2       Social_event_attendance    0.037488
9              Alone_normalized    0.021040
3                 Going_outside    0.015305
6                Post_frequency    0.006948
5           Friends_circle_size    0.004796
7                 Missing_count    0.001881
8           Online_social_ratio    0.001462
                        feature  importance
1                    Stage_fear   17.990798
9              Alone_normalized   17.366829
4     Drained_after_socializing   13.278712
10                  Is_low_post    9.817655
0              Time_spent_Alone    9.352524
3                 Going_outside    8.519566
5           Friends_circle_size    5.057261
2       Social_event_attendance    4.748085
6                Post_frequency 

In [20]:
voting_clf = VotingClassifier(
    estimators=[
        ('cat', cat_model),
        ('lgb', lgb_model),
        ('xgb', xgb_model),
        ('rf', rf_model)
    ],
    voting='soft'  # or 'hard'
)

voting_clf.fit(train, y)

test_pred = voting_clf.predict(test)
print("Finished")

0:	learn: 0.4870800	total: 1.85ms	remaining: 195ms
1:	learn: 0.3659073	total: 3.72ms	remaining: 193ms
2:	learn: 0.2898821	total: 5.03ms	remaining: 173ms
3:	learn: 0.2394434	total: 6.26ms	remaining: 160ms
4:	learn: 0.2051688	total: 7.18ms	remaining: 145ms
5:	learn: 0.1812125	total: 8.32ms	remaining: 139ms
6:	learn: 0.1648519	total: 9.78ms	remaining: 138ms
7:	learn: 0.1532523	total: 10.9ms	remaining: 134ms
8:	learn: 0.1455545	total: 12.3ms	remaining: 133ms
9:	learn: 0.1404007	total: 13.9ms	remaining: 133ms
10:	learn: 0.1367551	total: 15.4ms	remaining: 133ms
11:	learn: 0.1343020	total: 16.9ms	remaining: 132ms
12:	learn: 0.1324961	total: 18.3ms	remaining: 131ms
13:	learn: 0.1312941	total: 19.6ms	remaining: 129ms
14:	learn: 0.1305300	total: 21ms	remaining: 127ms
15:	learn: 0.1300211	total: 22.7ms	remaining: 128ms
16:	learn: 0.1295588	total: 24.1ms	remaining: 126ms
17:	learn: 0.1287819	total: 25.4ms	remaining: 124ms
18:	learn: 0.1285440	total: 26.4ms	remaining: 121ms
19:	learn: 0.1285420	tot

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [12:01:23] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Finished


In [21]:
test_pred = pd.Series(test_pred).map({1: 'Extrovert', 0: 'Introvert'})
submission = pd.DataFrame({
        "id": test_ID,
        'Personality': test_pred,
    })
submission.to_csv("submission11.csv", index=False)

In [23]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

xgb_model = xgb.XGBClassifier(**study_xgb.best_params, use_label_encoder=False)
cat_model = CatBoostClassifier(**study_cat.best_params)
lgb_model = LGBMClassifier(**study_lgb.best_params)
rf_model = RandomForestClassifier(**grid.best_params_)

xgb_model.fit(X_train, y_train)       
cat_model.fit(X_train, y_train)       
lgb_model.fit(X_train, y_train)      
rf_model.fit(X_train, y_train)

print("Finished")

# 二阶模型（你也可以换成 XGBClassifier）
final_estimator = LogisticRegression(max_iter=1000)

# 构建 StackingClassifier
stack_model = StackingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('cat', cat_model),
        ('lgb', lgb_model),
        ('rf', rf_model)
    ],
    final_estimator=final_estimator,
    passthrough=False,  # 是否保留原始特征（推荐 False，先试）
    cv=5,                # 内部交叉验证拆分
    #n_jobs=-1            # 并行训练
)


# 模型训练
stack_model.fit(X_train, y_train)

# 预测与评估
y_pred = stack_model.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print("Stacking 准确率：", acc)


/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [12:22:05] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0:	learn: 0.4893616	total: 1.75ms	remaining: 184ms
1:	learn: 0.3678428	total: 3.01ms	remaining: 156ms
2:	learn: 0.2915337	total: 4.07ms	remaining: 140ms
3:	learn: 0.2404997	total: 4.95ms	remaining: 126ms
4:	learn: 0.2058363	total: 5.63ms	remaining: 114ms
5:	learn: 0.1825248	total: 6.72ms	remaining: 112ms
6:	learn: 0.1664121	total: 7.8ms	remaining: 110ms
7:	learn: 0.1551359	total: 8.87ms	remaining: 109ms
8:	learn: 0.1465304	total: 9.66ms	remaining: 104ms
9:	learn: 0.1406897	total: 10.8ms	remaining: 104ms
10:	learn: 0.1365564	total: 12.1ms	remaining: 105ms
11:	learn: 0.1337150	total: 13.4ms	remaining: 105ms
12:	learn: 0.1318731	total: 14.6ms	remaining: 104ms
13:	learn: 0.1305866	total: 16ms	remaining: 105ms
14:	learn: 0.1296101	total: 17.2ms	remaining: 104ms
15:	learn: 0.1288405	total: 18.4ms	remaining: 103ms
16:	learn: 0.1282616	total: 19.5ms	remaining: 102ms
17:	learn: 0.1275869	total: 20.7ms	remaining: 101ms
18:	learn: 0.1273324	total: 21.4ms	remaining: 97.8ms
19:	learn: 0.1268872	tot

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [12:22:06] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


45:	learn: 0.1235552	total: 55.4ms	remaining: 72.3ms
46:	learn: 0.1235534	total: 56.2ms	remaining: 70.6ms
47:	learn: 0.1235519	total: 57ms	remaining: 68.9ms
48:	learn: 0.1235349	total: 57.9ms	remaining: 67.4ms
49:	learn: 0.1235335	total: 58.7ms	remaining: 65.8ms
50:	learn: 0.1235320	total: 60ms	remaining: 64.7ms
51:	learn: 0.1235306	total: 61.7ms	remaining: 64.1ms
52:	learn: 0.1235010	total: 64.1ms	remaining: 64.1ms
53:	learn: 0.1234995	total: 66.6ms	remaining: 64.1ms
54:	learn: 0.1234110	total: 70.1ms	remaining: 65ms
55:	learn: 0.1234095	total: 72.7ms	remaining: 64.9ms
56:	learn: 0.1234081	total: 73.9ms	remaining: 63.6ms
57:	learn: 0.1234067	total: 74.7ms	remaining: 61.9ms
58:	learn: 0.1234053	total: 75.9ms	remaining: 60.4ms
59:	learn: 0.1234039	total: 77ms	remaining: 59ms
60:	learn: 0.1234025	total: 78.3ms	remaining: 57.7ms
61:	learn: 0.1234011	total: 79.5ms	remaining: 56.4ms
62:	learn: 0.1233998	total: 80.7ms	remaining: 55.1ms
63:	learn: 0.1233984	total: 81.9ms	remaining: 53.8ms
64:

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [12:22:07] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [12:22:07] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [12:22:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [12:22:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-pack

0:	learn: 0.4897251	total: 1.43ms	remaining: 150ms
1:	learn: 0.3681465	total: 2.81ms	remaining: 146ms
2:	learn: 0.2932556	total: 4.03ms	remaining: 139ms
3:	learn: 0.2418576	total: 4.88ms	remaining: 125ms
4:	learn: 0.2059040	total: 5.49ms	remaining: 111ms
5:	learn: 0.1831262	total: 6.53ms	remaining: 109ms
6:	learn: 0.1668107	total: 7.56ms	remaining: 107ms
7:	learn: 0.1550621	total: 8.9ms	remaining: 109ms
8:	learn: 0.1464277	total: 9.89ms	remaining: 107ms
9:	learn: 0.1404346	total: 11ms	remaining: 105ms
10:	learn: 0.1363886	total: 12ms	remaining: 103ms
11:	learn: 0.1331949	total: 13ms	remaining: 102ms
12:	learn: 0.1310891	total: 14ms	remaining: 100ms
13:	learn: 0.1297293	total: 15.1ms	remaining: 99.2ms
14:	learn: 0.1284715	total: 16.2ms	remaining: 98.5ms
15:	learn: 0.1275339	total: 17.2ms	remaining: 96.9ms
16:	learn: 0.1267552	total: 18.3ms	remaining: 95.8ms
17:	learn: 0.1261362	total: 19.7ms	remaining: 96.3ms
18:	learn: 0.1254819	total: 20.8ms	remaining: 95.2ms
19:	learn: 0.1250085	tota

In [27]:
final_estimator = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    n_estimators=100,
    max_depth=3
)

stack_model = StackingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('cat', cat_model),
        ('lgb', lgb_model),
        ('rf', rf_model)
    ],
    final_estimator=final_estimator,
    passthrough=False,  # 是否保留原始特征（推荐 False，先试）
    cv=5,                # 内部交叉验证拆分
    #n_jobs=-1            # 并行训练
)


# 模型训练
stack_model.fit(X_train, y_train)

# 预测与评估
y_pred = stack_model.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print("Stacking 准确率：", acc)

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:38:53] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0:	learn: 0.5999684	total: 733us	remaining: 86.5ms
1:	learn: 0.5259349	total: 1.25ms	remaining: 73.3ms
2:	learn: 0.4655340	total: 2.09ms	remaining: 81ms
3:	learn: 0.4155882	total: 2.9ms	remaining: 83.4ms
4:	learn: 0.3738200	total: 3.71ms	remaining: 84.7ms
5:	learn: 0.3384344	total: 4.54ms	remaining: 85.4ms
6:	learn: 0.3086183	total: 5.35ms	remaining: 85.7ms
7:	learn: 0.2832766	total: 6.17ms	remaining: 85.6ms
8:	learn: 0.2616382	total: 8.41ms	remaining: 103ms
9:	learn: 0.2429745	total: 9.25ms	remaining: 101ms
10:	learn: 0.2270921	total: 10.1ms	remaining: 99.1ms
11:	learn: 0.2130903	total: 10.9ms	remaining: 97.4ms
12:	learn: 0.2013186	total: 11.8ms	remaining: 96.2ms
13:	learn: 0.1910393	total: 12.4ms	remaining: 92.8ms
14:	learn: 0.1820078	total: 13.1ms	remaining: 90.5ms
15:	learn: 0.1744848	total: 13.9ms	remaining: 89.4ms
16:	learn: 0.1678723	total: 14.6ms	remaining: 87.3ms
17:	learn: 0.1620716	total: 15.4ms	remaining: 86.2ms
18:	learn: 0.1572715	total: 15.8ms	remaining: 83.4ms
19:	learn

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:38:55] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:38:55] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:38:55] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:38:55] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-pack

0:	learn: 0.5988876	total: 609us	remaining: 71.9ms
1:	learn: 0.5239698	total: 1.39ms	remaining: 81.6ms
2:	learn: 0.4634957	total: 2.24ms	remaining: 86.6ms
3:	learn: 0.4136049	total: 3.03ms	remaining: 87.2ms
4:	learn: 0.3719814	total: 3.72ms	remaining: 84.8ms
5:	learn: 0.3368066	total: 4.55ms	remaining: 85.7ms
6:	learn: 0.3069319	total: 5.47ms	remaining: 87.6ms
7:	learn: 0.2814258	total: 6.31ms	remaining: 87.6ms
8:	learn: 0.2598081	total: 7.1ms	remaining: 86.7ms
9:	learn: 0.2408517	total: 7.88ms	remaining: 85.9ms
10:	learn: 0.2250919	total: 8.78ms	remaining: 86.2ms
11:	learn: 0.2113435	total: 9.59ms	remaining: 85.5ms
12:	learn: 0.1994264	total: 10.5ms	remaining: 85.3ms
13:	learn: 0.1890508	total: 11.3ms	remaining: 84.4ms
14:	learn: 0.1803191	total: 12ms	remaining: 83.3ms
15:	learn: 0.1726549	total: 12.6ms	remaining: 81.2ms
16:	learn: 0.1660158	total: 13.4ms	remaining: 80.3ms
17:	learn: 0.1604064	total: 14.1ms	remaining: 79.3ms
18:	learn: 0.1554506	total: 14.9ms	remaining: 78.4ms
19:	lea

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:38:58] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [51]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

X_train, X_val, y_train, y_val = train_test_split(train, y, test_size=0.2, random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print("最佳参数：", grid.best_params_)
print("最佳准确率：", grid.best_score_)

最佳参数： {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 50}
最佳准确率： 0.9696339195557627


In [59]:
from catboost import CatBoostClassifier, Pool

def objective_xgb(trial):
    params = {
        "verbosity": 0,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
    }

    model = xgb.XGBClassifier(**params, use_label_encoder=False)
    scores = cross_val_score(model, train, y, cv=5, scoring='accuracy')
    best_features(model, 17)
    return scores.mean()

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=50)

print("Best XGBoost parameters:", study_xgb.best_params)

def objective_cat(trial):
    params = {
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'iterations': trial.suggest_int('iterations', 100, 500),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'bootstrap_type': trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS']),
        'random_state': 42,
        'verbose': 0
    }

    model = CatBoostClassifier(**params)
    score = cross_val_score(model, train, y, cv=5, scoring='accuracy').mean()
    best_features(model, 17)
    return score

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=50)

print("最佳参数（CatBoost）：", study_cat.best_params)

def objective_lgb(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42
    }

    model = LGBMClassifier(**params)
    score = cross_val_score(model, train, y, cv=5, scoring='accuracy').mean()
    best_features(model, 17)
    return score

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=50)

print("最佳参数（LightGBM）：", study_lgb.best_params)

[I 2025-07-19 10:17:11,405] A new study created in memory with name: no-name-e0c5610a-ad63-44f2-b177-5f98accb0d88
[I 2025-07-19 10:17:12,687] Trial 0 finished with value: 0.968959435471883 and parameters: {'max_depth': 3, 'learning_rate': 0.016459391948103863, 'n_estimators': 612, 'gamma': 2.1830039773283803, 'min_child_weight': 7, 'subsample': 0.8523739899989522, 'colsample_bytree': 0.9481463372827434, 'lambda': 0.03824872031150492, 'alpha': 6.4132100629786555}. Best is trial 0 with value: 0.968959435471883.
[I 2025-07-19 10:17:13,757] Trial 1 finished with value: 0.9693913426197159 and parameters: {'max_depth': 4, 'learning_rate': 0.13041242826073887, 'n_estimators': 715, 'gamma': 2.1784772256354623, 'min_child_weight': 2, 'subsample': 0.9101886039005476, 'colsample_bytree': 0.6982899378106298, 'lambda': 0.03316567030453854, 'alpha': 1.7738953840967968}. Best is trial 1 with value: 0.9693913426197159.
[I 2025-07-19 10:17:17,162] Trial 2 finished with value: 0.9691753890457994 and par

Best XGBoost parameters: {'max_depth': 7, 'learning_rate': 0.029828785576639597, 'n_estimators': 243, 'gamma': 2.9177542715207796, 'min_child_weight': 5, 'subsample': 0.7420911443196798, 'colsample_bytree': 0.7716568498241789, 'lambda': 0.5996406267495297, 'alpha': 0.7354918346534538}


[I 2025-07-19 10:19:12,839] Trial 0 finished with value: 0.9679337652987762 and parameters: {'depth': 9, 'learning_rate': 0.13061614380349076, 'iterations': 270, 'l2_leaf_reg': 4.313386527959576, 'bootstrap_type': 'MVS'}. Best is trial 0 with value: 0.9679337652987762.
[I 2025-07-19 10:19:16,212] Trial 1 finished with value: 0.9652345933782789 and parameters: {'depth': 7, 'learning_rate': 0.2557743484913612, 'iterations': 250, 'l2_leaf_reg': 0.7917089772993882, 'bootstrap_type': 'Bernoulli'}. Best is trial 0 with value: 0.9679337652987762.
[I 2025-07-19 10:19:19,210] Trial 2 finished with value: 0.9670160427651618 and parameters: {'depth': 4, 'learning_rate': 0.2633822279834094, 'iterations': 365, 'l2_leaf_reg': 0.011908270346329934, 'bootstrap_type': 'Bayesian'}. Best is trial 0 with value: 0.9679337652987762.
[I 2025-07-19 10:19:21,530] Trial 3 finished with value: 0.968095723192347 and parameters: {'depth': 4, 'learning_rate': 0.19052424848641406, 'iterations': 284, 'l2_leaf_reg': 0

最佳参数（CatBoost）： {'depth': 6, 'learning_rate': 0.026321039324909205, 'iterations': 128, 'l2_leaf_reg': 0.17812278723198025, 'bootstrap_type': 'Bayesian'}
[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000586 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 225
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.739524 -> initscore=1.043494
[LightGBM] [Info] Start training from score 1.043494
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of

[I 2025-07-19 10:21:32,039] Trial 0 finished with value: 0.9679338527411734 and parameters: {'num_leaves': 40, 'max_depth': 8, 'learning_rate': 0.12300928498890912, 'n_estimators': 183, 'min_child_samples': 13, 'subsample': 0.5143888638319687, 'colsample_bytree': 0.7867007217827007}. Best is trial 0 with value: 0.9679338527411734.


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-07-19 10:22:16,637] Trial 48 finished with value: 0.9691754181932651 and parameters: {'num_leaves': 59, 'max_depth': 5, 'learning_rate': 0.02308095401969238, 'n_estimators': 324, 'min_child_samples': 12, 'subsample': 0.5734765363210853, 'colsample_bytree': 0.8469356082478684}. Best is trial 11 with value: 0.9695532567920881.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000444 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 225
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.739

[I 2025-07-19 10:22:17,315] Trial 49 finished with value: 0.9690134602996942 and parameters: {'num_leaves': 63, 'max_depth': 4, 'learning_rate': 0.0715377585842727, 'n_estimators': 280, 'min_child_samples': 10, 'subsample': 0.5326697047708747, 'colsample_bytree': 0.8809866674441653}. Best is trial 11 with value: 0.9695532567920881.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [60]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 4, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'random_state': 42
    }

    model = RandomForestClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, train, y, cv=cv, scoring='accuracy')
    return scores.mean()

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=50)

print("Best RF params:", study_rf.best_params)


[I 2025-07-19 10:27:33,476] A new study created in memory with name: no-name-505d9dce-7e95-47a1-8d2d-89f27f2702bc
[I 2025-07-19 10:27:48,580] Trial 0 finished with value: 0.9692292389888161 and parameters: {'n_estimators': 379, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 'log2', 'bootstrap': True}. Best is trial 0 with value: 0.9692292389888161.
[I 2025-07-19 10:28:03,575] Trial 1 finished with value: 0.9690672956689781 and parameters: {'n_estimators': 381, 'max_depth': 14, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: 0.9692292389888161.
[I 2025-07-19 10:28:17,499] Trial 2 finished with value: 0.9690133145623653 and parameters: {'n_estimators': 421, 'max_depth': 7, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 'log2', 'bootstrap': False}. Best is trial 0 with value: 0.9692292389888161.
[I 2025-07-19 10:28:45,817] Trial 3 finished with value: 0.9683654829880816 

Best RF params: {'n_estimators': 224, 'max_depth': 16, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': True}


In [64]:
xgb_model = xgb.XGBClassifier(**study_xgb.best_params, use_label_encoder=False)
cat_model = CatBoostClassifier(**study_cat.best_params)
lgb_model = LGBMClassifier(**study_lgb.best_params)
rf_model = RandomForestClassifier(**grid.best_params_)


xgb_model.fit(X_train, y_train)       
cat_model.fit(X_train, y_train)       
lgb_model.fit(X_train, y_train)      
rf_model.fit(X_train, y_train)

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [10:51:42] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0:	learn: 0.6702991	total: 1.14ms	remaining: 145ms
1:	learn: 0.6487231	total: 2.36ms	remaining: 149ms
2:	learn: 0.6281967	total: 3.09ms	remaining: 129ms
3:	learn: 0.6086883	total: 3.89ms	remaining: 121ms
4:	learn: 0.5899777	total: 4.82ms	remaining: 119ms
5:	learn: 0.5721624	total: 5.56ms	remaining: 113ms
6:	learn: 0.5554239	total: 6.41ms	remaining: 111ms
7:	learn: 0.5392053	total: 7.22ms	remaining: 108ms
8:	learn: 0.5238493	total: 8.05ms	remaining: 106ms
9:	learn: 0.5091193	total: 8.86ms	remaining: 105ms
10:	learn: 0.4949979	total: 9.67ms	remaining: 103ms
11:	learn: 0.4814562	total: 10.4ms	remaining: 101ms
12:	learn: 0.4684771	total: 11.3ms	remaining: 99.6ms
13:	learn: 0.4560177	total: 12.4ms	remaining: 101ms
14:	learn: 0.4441325	total: 13.4ms	remaining: 101ms
15:	learn: 0.4326692	total: 14.4ms	remaining: 101ms
16:	learn: 0.4216879	total: 15.3ms	remaining: 100ms
17:	learn: 0.4111592	total: 16.4ms	remaining: 100ms
18:	learn: 0.4012021	total: 17ms	remaining: 97.8ms
19:	learn: 0.3914303	t

,n_estimators,100
,criterion,'gini'
,max_depth,10
,min_samples_split,5
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [72]:
voting_clf = VotingClassifier(
    estimators=[
        ('cat', cat_model),
        ('lgb', lgb_model),
        ('xgb', xgb_model),
        ('rf', rf_model)
    ],
    voting='soft'  # or 'hard'
)

voting_clf.fit(X_train, y_train)

y_pred = voting_clf.predict(X_val)
print("投票器准确率：", accuracy_score(y_val, y_pred))

0:	learn: 0.6702991	total: 1.66ms	remaining: 211ms
1:	learn: 0.6487231	total: 2.82ms	remaining: 178ms
2:	learn: 0.6281967	total: 3.8ms	remaining: 159ms
3:	learn: 0.6086883	total: 4.93ms	remaining: 153ms
4:	learn: 0.5899777	total: 6.11ms	remaining: 150ms
5:	learn: 0.5721624	total: 7.06ms	remaining: 144ms
6:	learn: 0.5554239	total: 8.08ms	remaining: 140ms
7:	learn: 0.5392053	total: 9.07ms	remaining: 136ms
8:	learn: 0.5238493	total: 10.1ms	remaining: 133ms
9:	learn: 0.5091193	total: 11ms	remaining: 129ms
10:	learn: 0.4949979	total: 11.9ms	remaining: 126ms
11:	learn: 0.4814562	total: 12.7ms	remaining: 123ms
12:	learn: 0.4684771	total: 13.7ms	remaining: 121ms
13:	learn: 0.4560177	total: 14.7ms	remaining: 120ms
14:	learn: 0.4441325	total: 15.6ms	remaining: 118ms
15:	learn: 0.4326692	total: 16.6ms	remaining: 116ms
16:	learn: 0.4216879	total: 17.6ms	remaining: 115ms
17:	learn: 0.4111592	total: 18.5ms	remaining: 113ms
18:	learn: 0.4012021	total: 19.1ms	remaining: 110ms
19:	learn: 0.3914303	tota

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:02:43] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


投票器准确率： 0.9686909581646423


In [74]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 二阶模型（你也可以换成 XGBClassifier）
final_estimator = LogisticRegression(max_iter=1000)

# 构建 StackingClassifier
stack_model = StackingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('cat', cat_model),
        ('lgb', lgb_model),
        ('rf', rf_model)
    ],
    final_estimator=final_estimator,
    passthrough=False,  # 是否保留原始特征（推荐 False，先试）
    cv=5,                # 内部交叉验证拆分
    #n_jobs=-1            # 并行训练
)


# 模型训练
stack_model.fit(X_train, y_train)

# 预测与评估
y_pred = stack_model.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print("Stacking 准确率：", acc)

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:04:09] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0:	learn: 0.6702991	total: 1.4ms	remaining: 178ms
1:	learn: 0.6487231	total: 2.59ms	remaining: 163ms
2:	learn: 0.6281967	total: 3.48ms	remaining: 145ms
3:	learn: 0.6086883	total: 4.43ms	remaining: 137ms
4:	learn: 0.5899777	total: 5.32ms	remaining: 131ms
5:	learn: 0.5721624	total: 6.28ms	remaining: 128ms
6:	learn: 0.5554239	total: 7.2ms	remaining: 125ms
7:	learn: 0.5392053	total: 8.06ms	remaining: 121ms
8:	learn: 0.5238493	total: 9.01ms	remaining: 119ms
9:	learn: 0.5091193	total: 9.98ms	remaining: 118ms
10:	learn: 0.4949979	total: 10.9ms	remaining: 116ms
11:	learn: 0.4814562	total: 11.7ms	remaining: 113ms
12:	learn: 0.4684771	total: 12.6ms	remaining: 111ms
13:	learn: 0.4560177	total: 13.5ms	remaining: 110ms
14:	learn: 0.4441325	total: 14.5ms	remaining: 109ms
15:	learn: 0.4326692	total: 15.4ms	remaining: 108ms
16:	learn: 0.4216879	total: 16.3ms	remaining: 106ms
17:	learn: 0.4111592	total: 17.2ms	remaining: 105ms
18:	learn: 0.4012021	total: 17.9ms	remaining: 102ms
19:	learn: 0.3914303	tot

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:04:10] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:04:10] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:04:10] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:04:10] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-pack

0:	learn: 0.6702994	total: 1.06ms	remaining: 135ms
1:	learn: 0.6487152	total: 2.06ms	remaining: 130ms
2:	learn: 0.6280323	total: 2.76ms	remaining: 115ms
3:	learn: 0.6083468	total: 3.54ms	remaining: 110ms
4:	learn: 0.5895832	total: 4.29ms	remaining: 106ms
5:	learn: 0.5717415	total: 5.06ms	remaining: 103ms
6:	learn: 0.5549204	total: 5.86ms	remaining: 101ms
7:	learn: 0.5386742	total: 6.62ms	remaining: 99.2ms
8:	learn: 0.5232704	total: 7.41ms	remaining: 98ms
9:	learn: 0.5085042	total: 8.12ms	remaining: 95.8ms
10:	learn: 0.4943489	total: 9.27ms	remaining: 98.6ms
11:	learn: 0.4807785	total: 10ms	remaining: 96.9ms
12:	learn: 0.4677621	total: 11ms	remaining: 97.2ms
13:	learn: 0.4552752	total: 11.8ms	remaining: 95.8ms
14:	learn: 0.4433901	total: 12.6ms	remaining: 94.7ms
15:	learn: 0.4319251	total: 13.4ms	remaining: 93.5ms
16:	learn: 0.4209227	total: 14.3ms	remaining: 93.1ms
17:	learn: 0.4103642	total: 15ms	remaining: 91.7ms
18:	learn: 0.4003788	total: 15.6ms	remaining: 89.3ms
19:	learn: 0.39058

In [83]:
xgb_model = xgb.XGBClassifier(**study_xgb.best_params, use_label_encoder=False)
cat_model = CatBoostClassifier(**study_cat.best_params)
lgb_model = LGBMClassifier(**study_lgb.best_params)
rf_model = RandomForestClassifier(**grid.best_params_)


xgb_model.fit(train, y)       
cat_model.fit(train, y)       
lgb_model.fit(train, y)      
rf_model.fit(train, y)

print("Finished")

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:16:39] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0:	learn: 0.6704037	total: 2.99ms	remaining: 379ms
1:	learn: 0.6488328	total: 4.34ms	remaining: 274ms
2:	learn: 0.6282247	total: 5.38ms	remaining: 224ms
3:	learn: 0.6086046	total: 7.46ms	remaining: 231ms
4:	learn: 0.5899148	total: 8.48ms	remaining: 209ms
5:	learn: 0.5721508	total: 9.47ms	remaining: 193ms
6:	learn: 0.5553186	total: 10.5ms	remaining: 181ms
7:	learn: 0.5391284	total: 11.4ms	remaining: 171ms
8:	learn: 0.5237378	total: 12.4ms	remaining: 164ms
9:	learn: 0.5090319	total: 13.3ms	remaining: 157ms
10:	learn: 0.4949245	total: 14.3ms	remaining: 152ms
11:	learn: 0.4813986	total: 15.2ms	remaining: 147ms
12:	learn: 0.4684400	total: 16.1ms	remaining: 142ms
13:	learn: 0.4560182	total: 17.1ms	remaining: 139ms
14:	learn: 0.4441254	total: 18ms	remaining: 136ms
15:	learn: 0.4326520	total: 19.1ms	remaining: 133ms
16:	learn: 0.4216786	total: 20ms	remaining: 131ms
17:	learn: 0.4111590	total: 21ms	remaining: 128ms
18:	learn: 0.4010993	total: 21.9ms	remaining: 126ms
19:	learn: 0.3913729	total: 

In [84]:
final_estimator = LogisticRegression(max_iter=1000)

# 构建 StackingClassifier
stack_model = StackingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('cat', cat_model),
        ('lgb', lgb_model)
    ],
    final_estimator=final_estimator,
    passthrough=False,  # 是否保留原始特征（推荐 False，先试）
    cv=5,                # 内部交叉验证拆分
    #n_jobs=-1            # 并行训练
)


# 模型训练
stack_model.fit(X_train, y_train)

# 预测与评估
y_pred = stack_model.predict(test)
#test = construct_features(test)
print(test.shape)

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:17:36] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0:	learn: 0.6702991	total: 1.28ms	remaining: 162ms
1:	learn: 0.6487231	total: 2.32ms	remaining: 146ms
2:	learn: 0.6281967	total: 3.02ms	remaining: 126ms
3:	learn: 0.6086883	total: 3.84ms	remaining: 119ms
4:	learn: 0.5899777	total: 4.7ms	remaining: 116ms
5:	learn: 0.5721624	total: 5.48ms	remaining: 111ms
6:	learn: 0.5554239	total: 6.33ms	remaining: 110ms
7:	learn: 0.5392053	total: 8.43ms	remaining: 126ms
8:	learn: 0.5238493	total: 9.62ms	remaining: 127ms
9:	learn: 0.5091193	total: 10.6ms	remaining: 125ms
10:	learn: 0.4949979	total: 11.5ms	remaining: 123ms
11:	learn: 0.4814562	total: 12.4ms	remaining: 120ms
12:	learn: 0.4684771	total: 13.2ms	remaining: 116ms
13:	learn: 0.4560177	total: 14.1ms	remaining: 115ms
14:	learn: 0.4441325	total: 16.6ms	remaining: 125ms
15:	learn: 0.4326692	total: 17.6ms	remaining: 123ms
16:	learn: 0.4216879	total: 18.4ms	remaining: 120ms
17:	learn: 0.4111592	total: 19.2ms	remaining: 117ms
18:	learn: 0.4012021	total: 19.8ms	remaining: 114ms
19:	learn: 0.3914303	to

/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:17:36] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:17:37] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:17:37] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [11:17:37] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/wangzeming/.local/lib/python3.10/site-pack

0:	learn: 0.6702994	total: 4.64ms	remaining: 589ms
1:	learn: 0.6487152	total: 7.02ms	remaining: 442ms
2:	learn: 0.6280323	total: 8.77ms	remaining: 366ms
3:	learn: 0.6083468	total: 12.7ms	remaining: 395ms
4:	learn: 0.5895832	total: 14.4ms	remaining: 354ms
5:	learn: 0.5717415	total: 15.9ms	remaining: 324ms
6:	learn: 0.5549204	total: 17.5ms	remaining: 302ms
7:	learn: 0.5386742	total: 19.1ms	remaining: 286ms
8:	learn: 0.5232704	total: 20.6ms	remaining: 272ms
9:	learn: 0.5085042	total: 21.8ms	remaining: 257ms
10:	learn: 0.4943489	total: 23.1ms	remaining: 245ms
11:	learn: 0.4807785	total: 24.3ms	remaining: 235ms
12:	learn: 0.4677621	total: 25.6ms	remaining: 226ms
13:	learn: 0.4552752	total: 27ms	remaining: 220ms
14:	learn: 0.4433901	total: 28.5ms	remaining: 215ms
15:	learn: 0.4319251	total: 29.7ms	remaining: 208ms
16:	learn: 0.4209227	total: 30.8ms	remaining: 201ms
17:	learn: 0.4103642	total: 31.9ms	remaining: 195ms
18:	learn: 0.4003788	total: 32.7ms	remaining: 187ms
19:	learn: 0.3905801	tot

In [85]:
print(X_train.shape)

(14819, 17)


In [86]:
test_pred = pd.Series(y_pred).map({1: 'Extrovert', 0: 'Introvert'})
submission = pd.DataFrame({
        "id": test_ID,
        'Personality': test_pred,
    })
submission.to_csv("submission11.csv", index=False)